In [1]:
import pandas as pd
import numpy as np

print("✅ Libraries imported")


✅ Libraries imported


In [2]:
# Load dataset 
df = pd.read_csv("email_phishing_dataset_FINAL.csv")

print("✅ Dataset loaded successfully")

✅ Dataset loaded successfully


In [3]:
# View first few rows
df.head()

,email_subject_len,email_has_urgent_keyword,email_from_domain,web_url,web_url_len,web_ip_add,web_geo_loc,web_tld,web_who_is,web_https,...,content_entropy,domain_trust_score,email_domain_matches_url,email_url_domain_similarity,content_num_forms,content_num_inputs,content_num_scripts,content_suspicious_keywords,semantic_coherence_score,brand_consistency_score
0,32,0,spamassassin.zones.apache.org,http://tools.ietf.org/html/rfc1583,34,30.180.42.35,United States,org,complete,yes,...,4.620961,0.8,0,0.411765,0,0,5,0,0.043153,0.5
1,46,0,gmail.com>,http://www.quickfixgolf.com,27,150.66.16.42,Japan,com,complete,yes,...,4.742243,0.8,0,0.500000,0,0,2,0,0.081125,0.5
2,21,0,telefonica.net>,http://www.lvnazarene.org,25,180.123.185.229,China,org,complete,yes,...,4.663432,0.8,0,0.400000,0,0,2,0,0.000000,0.5
3,99,1,gmail.com>,http://hatchersmartialarts.homestead.com/front...,51,46.97.122.170,Romania,com,complete,yes,...,4.971977,0.8,0,0.466667,0,1,2,2,0.071720,0.5
4,72,1,luebeck.de>,http://www.gabile.com/,22,94.145.85.24,Denmark,com,incomplete,no,...,4.338266,0.4,0,0.357143,0,1,24,3,0.000000,0.5


In [4]:
# Dataset shape
print("Dataset shape:", df.shape)


Dataset shape: (8000, 39)


In [5]:
# Check data types
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8000 entries, 0 to 7999
Data columns (total 39 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   email_subject_len            8000 non-null   int64  
 1   email_has_urgent_keyword     8000 non-null   int64  
 2   email_from_domain            7822 non-null   object 
 3   web_url                      8000 non-null   object 
 4   web_url_len                  8000 non-null   int64  
 5   web_ip_add                   8000 non-null   object 
 6   web_geo_loc                  8000 non-null   object 
 7   web_tld                      8000 non-null   object 
 8   web_who_is                   8000 non-null   object 
 9   web_https                    8000 non-null   object 
 10  web_js_len                   8000 non-null   float64
 11  web_js_obf_len               8000 non-null   float64
 12  web_content                  8000 non-null   object 
 13  domain_age        

In [3]:
# email_from_domain missing values
df = df.dropna(subset=['email_from_domain'])

print("Shape after dropping email_from_domain missing:", df.shape)


Shape after dropping email_from_domain missing: (7822, 39)


In [5]:
# Create missingness indicator
df['domain_age_missing'] = df['domain_age'].isnull().astype(int)

# Impute missing domain_age with 0 (unknown/new domain)
df['domain_age'] = df['domain_age'].fillna(0)

print("Missing domain_age after handling:", df['domain_age'].isnull().sum())


Missing domain_age after handling: 0


In [8]:
print("\nFinal dataset shape:", df.shape)
print("\nAny remaining missing values?")
print(df.isnull().sum().sum())



Final dataset shape: (7822, 40)

Any remaining missing values?
0


In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 7822 entries, 0 to 7999
Data columns (total 40 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   email_subject_len            7822 non-null   int64  
 1   email_has_urgent_keyword     7822 non-null   int64  
 2   email_from_domain            7822 non-null   object 
 3   web_url                      7822 non-null   object 
 4   web_url_len                  7822 non-null   int64  
 5   web_ip_add                   7822 non-null   object 
 6   web_geo_loc                  7822 non-null   object 
 7   web_tld                      7822 non-null   object 
 8   web_who_is                   7822 non-null   object 
 9   web_https                    7822 non-null   object 
 10  web_js_len                   7822 non-null   float64
 11  web_js_obf_len               7822 non-null   float64
 12  web_content                  7822 non-null   object 
 13  domain_age             

In [6]:
# ENCODING NON-NUMERICAL FEATURES

df_encoded = df.copy()

# 1️⃣ email_from_domain → frequency encoding
domain_freq = df_encoded['email_from_domain'].value_counts(normalize=True)
df_encoded['email_domain_freq'] = df_encoded['email_from_domain'].map(domain_freq)
df_encoded['email_domain_freq'] = df_encoded['email_domain_freq'].fillna(0)
df_encoded.drop(['email_from_domain'], axis=1, inplace=True)

# 2️⃣ web_geo_loc → frequency encoding
geo_freq = df_encoded['web_geo_loc'].value_counts(normalize=True)
df_encoded['geo_freq'] = df_encoded['web_geo_loc'].map(geo_freq).fillna(0)
df_encoded.drop(['web_geo_loc'], axis=1, inplace=True)

# 3️⃣ web_tld → trust score
trusted_tlds = ['com', 'org', 'net', 'edu', 'gov']
df_encoded['tld_trust_score'] = df_encoded['web_tld'].apply(
    lambda x: 1.0 if x in trusted_tlds else 0.0
)
df_encoded.drop(['web_tld'], axis=1, inplace=True)

# 4️⃣ web_who_is → binary
df_encoded['web_who_is'] = df_encoded['web_who_is'].map({'complete': 1, 'incomplete': 0})

# 5️⃣ web_https → binary
df_encoded['web_https'] = df_encoded['web_https'].map({'yes': 1, 'no': 0})

# 6️⃣ Drop noisy features
df_encoded.drop(['web_url', 'web_ip_add', 'web_content'], axis=1, inplace=True)

print("✅ Encoding complete")
print("Final shape:", df_encoded.shape)


✅ Encoding complete
Final shape: (7822, 37)


In [11]:
df.head()

,email_subject_len,email_has_urgent_keyword,email_from_domain,web_url,web_url_len,web_ip_add,web_geo_loc,web_tld,web_who_is,web_https,...,domain_trust_score,email_domain_matches_url,email_url_domain_similarity,content_num_forms,content_num_inputs,content_num_scripts,content_suspicious_keywords,semantic_coherence_score,brand_consistency_score,domain_age_missing
0,32,0,spamassassin.zones.apache.org,http://tools.ietf.org/html/rfc1583,34,30.180.42.35,United States,org,complete,yes,...,0.8,0,0.411765,0,0,5,0,0.043153,0.5,0
1,46,0,gmail.com>,http://www.quickfixgolf.com,27,150.66.16.42,Japan,com,complete,yes,...,0.8,0,0.500000,0,0,2,0,0.081125,0.5,0
2,21,0,telefonica.net>,http://www.lvnazarene.org,25,180.123.185.229,China,org,complete,yes,...,0.8,0,0.400000,0,0,2,0,0.000000,0.5,0
3,99,1,gmail.com>,http://hatchersmartialarts.homestead.com/front...,51,46.97.122.170,Romania,com,complete,yes,...,0.8,0,0.466667,0,1,2,2,0.071720,0.5,0
4,72,1,luebeck.de>,http://www.gabile.com/,22,94.145.85.24,Denmark,com,incomplete,no,...,0.4,0,0.357143,0,1,24,3,0.000000,0.5,0


In [12]:
df_encoded.head()

,email_subject_len,email_has_urgent_keyword,web_url_len,web_who_is,web_https,web_js_len,web_js_obf_len,domain_age,final_label,js_obfuscation_ratio,...,content_num_forms,content_num_inputs,content_num_scripts,content_suspicious_keywords,semantic_coherence_score,brand_consistency_score,domain_age_missing,email_domain_freq,geo_freq,tld_trust_score
0,32,0,34,1,1,137.0,0.00,11168.0,0,0.000000,...,0,0,5,0,0.043153,0.5,0,0.001278,0.424572,1.0
1,46,0,27,1,1,94.0,0.00,9692.0,0,0.000000,...,0,0,2,0,0.081125,0.5,0,0.071976,0.057658,1.0
2,21,0,25,1,1,44.5,0.00,2344.0,0,0.000000,...,0,0,2,0,0.000000,0.5,0,0.006009,0.094861,1.0
3,99,1,51,1,1,84.5,0.00,10335.0,0,0.000000,...,0,1,2,2,0.071720,0.5,0,0.071976,0.002046,1.0
4,72,1,22,0,0,837.0,460.35,7421.0,1,0.549344,...,0,1,24,3,0.000000,0.5,0,0.000128,0.004858,1.0


In [13]:
print(df_encoded.head())

   email_subject_len  email_has_urgent_keyword  web_url_len  web_who_is  \
0                 32                         0           34           1   
1                 46                         0           27           1   
2                 21                         0           25           1   
3                 99                         1           51           1   
4                 72                         1           22           0   

   web_https  web_js_len  web_js_obf_len  domain_age  final_label  \
0          1       137.0            0.00     11168.0            0   
1          1        94.0            0.00      9692.0            0   
2          1        44.5            0.00      2344.0            0   
3          1        84.5            0.00     10335.0            0   
4          0       837.0          460.35      7421.0            1   

   js_obfuscation_ratio  ...  content_num_forms  content_num_inputs  \
0              0.000000  ...                  0                

In [7]:
from sklearn.model_selection import train_test_split

X = df_encoded.drop('final_label', axis=1)
y = df_encoded['final_label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print(X_train.shape, X_test.shape)


(6257, 36) (1565, 36)


In [7]:
# Count classes in training set
import pandas as pd

print("Training set class distribution:")
print(y_train.value_counts())

print("\nTest set class distribution:")
print(y_test.value_counts())


Training set class distribution:
final_label
0    3200
1    3057
Name: count, dtype: int64

Test set class distribution:
final_label
0    800
1    765
Name: count, dtype: int64


In [8]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

X_train_scaled = pd.DataFrame(X_train_scaled, columns=X_train.columns)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X_test.columns)


In [9]:
# First 5 rows of scaled training data
print("Scaled Training Data:")
display(X_train_scaled.head())

# First 5 rows of scaled test data
print("\nScaled Test Data:")
display(X_test_scaled.head())


Scaled Training Data:


,email_subject_len,email_has_urgent_keyword,web_url_len,web_who_is,web_https,web_js_len,web_js_obf_len,domain_age,js_obfuscation_ratio,url_has_ip,...,content_num_forms,content_num_inputs,content_num_scripts,content_suspicious_keywords,semantic_coherence_score,brand_consistency_score,domain_age_missing,email_domain_freq,geo_freq,tld_trust_score
0,0.136842,1.0,0.126354,1.0,1.0,0.152207,0.000000,0.547386,0.000000,0.0,...,0.0,0.142857,0.000000,0.25,0.322002,1.0,0.0,1.000000,1.000000,1.0
1,0.098246,1.0,0.003610,1.0,1.0,0.180307,0.000000,0.538528,0.000000,0.0,...,0.0,0.000000,0.083333,0.50,0.000000,1.0,0.0,1.000000,1.000000,0.0
2,0.087719,0.0,0.021661,0.0,0.0,0.539515,0.317010,0.035365,0.584526,0.0,...,0.0,0.142857,0.333333,0.25,0.782408,1.0,0.0,0.000000,0.069277,1.0
3,0.070175,1.0,0.212996,0.0,1.0,0.950474,0.934189,0.000000,0.978669,0.0,...,0.0,0.285714,0.541667,0.75,0.223506,1.0,1.0,0.000000,0.029819,1.0
4,0.189474,0.0,0.119134,1.0,1.0,0.206650,0.000000,0.014252,0.000000,0.0,...,0.0,0.000000,0.083333,0.75,0.000000,1.0,0.0,0.007117,1.000000,1.0



Scaled Test Data:


,email_subject_len,email_has_urgent_keyword,web_url_len,web_who_is,web_https,web_js_len,web_js_obf_len,domain_age,js_obfuscation_ratio,url_has_ip,...,content_num_forms,content_num_inputs,content_num_scripts,content_suspicious_keywords,semantic_coherence_score,brand_consistency_score,domain_age_missing,email_domain_freq,geo_freq,tld_trust_score
0,0.112281,1.0,0.043321,0.0,1.0,0.227725,0.000000,0.680386,0.000000,0.0,...,0.0,0.142857,0.166667,0.75,0.00000,1.0,0.0,0.007117,0.223193,1.0
1,0.014035,1.0,0.119134,0.0,0.0,0.730242,0.647518,0.013054,0.882602,0.0,...,0.0,0.428571,0.708333,0.75,0.29652,1.0,0.0,0.000000,1.000000,1.0
2,0.238596,1.0,0.057762,0.0,1.0,0.000000,0.000000,0.000000,0.000000,0.0,...,0.0,0.000000,0.000000,0.00,0.00000,1.0,1.0,0.026690,1.000000,1.0
3,0.122807,0.0,0.111913,0.0,0.0,0.839831,0.556276,0.000000,0.659431,0.0,...,0.0,0.571429,0.833333,0.75,0.47363,1.0,1.0,0.001779,1.000000,1.0
4,0.140351,0.0,0.090253,0.0,0.0,0.900948,0.837386,0.000000,0.925418,0.0,...,0.0,0.428571,0.708333,0.50,0.00000,1.0,1.0,0.001779,0.080723,1.0


In [9]:
#FEATURE SELECTION ON TRAIN DATA

from sklearn.feature_selection import mutual_info_classif

mi_scores = mutual_info_classif(
    X_train_scaled,
    y_train,
    random_state=42
)

mi_df = pd.DataFrame({
    'feature': X_train.columns,
    'mi_score': mi_scores
}).sort_values(by='mi_score', ascending=False)

mi_df.head(15)


,feature,mi_score
5,web_js_len,0.657304
8,js_obfuscation_ratio,0.521160
6,web_js_obf_len,0.519802
28,content_num_scripts,0.494982
22,content_entropy,0.472273
23,domain_trust_score,0.369185
33,email_domain_freq,0.324097
4,web_https,0.276868
29,content_suspicious_keywords,0.269552
3,web_who_is,0.261283


In [10]:
top_k = 6
top_features = mi_df.head(top_k)['feature'].tolist()

X_train_selected = X_train_scaled[top_features]
X_test_selected = X_test_scaled[top_features]

print("Selected features:", top_features)
print(X_train_selected.shape, X_test_selected.shape)


Selected features: ['web_js_len', 'js_obfuscation_ratio', 'web_js_obf_len', 'content_num_scripts', 'content_entropy', 'domain_trust_score']
(6257, 6) (1565, 6)


In [11]:
#CORRECTED IMPORTS

# Qiskit imports (CORRECTED for 2025)
from qiskit import QuantumCircuit
from qiskit.circuit.library import ZZFeatureMap, RealAmplitudes
from qiskit_algorithms.utils import algorithm_globals

from qiskit_machine_learning.kernels import FidelityQuantumKernel
from qiskit_machine_learning.algorithms import QSVC
from qiskit.primitives import Sampler
from qiskit_aer import AerSimulator

print("📦 All libraries imported successfully!")
print("✅ Using FidelityQuantumKernel (current API)")

📦 All libraries imported successfully!
✅ Using FidelityQuantumKernel (current API)


In [12]:
# BATCH-WISE SVM & QSVM (WITH SOFT PROBABILITY PREDICTIONS)

from sklearn.svm import SVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import accuracy_score
import time
from qiskit.visualization import circuit_drawer
from sklearn.utils import shuffle

algorithm_globals.random_seed = 42

feature_map = ZZFeatureMap(
    feature_dimension=top_k,
    reps=1,
    entanglement="linear"
)
quantum_kernel = FidelityQuantumKernel(feature_map=feature_map)

print(f"✅ Quantum kernel created")
print(f"   • Circuit depth: ~{feature_map.depth()}")

if top_k <= 11:
    print(f"\n🔬 Quantum Feature Map Circuit:")
    print(feature_map.decompose().draw(output='text'))
else:
    print(f"\n🔬 Quantum circuit created (too large to display: {top_k} qubits)")


BATCH_SIZE = 500

print(f"Number of batches: {len(X_train_selected) // BATCH_SIZE + 1}")

# SHUFFLE DATA BEFORE BATCHING
print("\n🔀 Shuffling training data to ensure balanced batches...")

X_train_shuffled, y_train_shuffled = shuffle(
    X_train_selected,
    y_train,
    random_state=42
)

print("✅ Data shuffled")

# SCALE DATA AGAIN WITH PI (SAME FOR BOTH MODELS)
print("\n📊 Scaling data for both SVM and QSVM...")

X_train_scaled_q = X_train_shuffled.values * np.pi
X_test_scaled_q  = X_test_selected.values  * np.pi

print("✅ Both models will use pi-scaled features")

# ── Storage lists ────────────────────────────────────────────────
svm_preds_list  = []   # hard predictions per batch
qsvm_preds_list = []   # hard predictions per batch

# NOTE: svm_proba_list is populated AFTER the loop via CalibratedClassifierCV
#       to avoid Platt-scaling CV competing with Aer's thread pool
qsvm_proba_list = []   # soft probabilities per batch (sigmoid on decision_function)

svm_models      = []   # trained SVM per batch (needed for post-hoc calibration)
qsvm_models     = []   # trained QSVM per batch

# Batch indices stored for calibration step
batch_indices   = []   # (start, end) index pairs of successful batches

total_batches   = 0
skipped_batches = 0

# ── Batch loop ───────────────────────────────────────────────────
print("STARTING BATCH-WISE TRAINING")

for i in range(0, len(X_train_shuffled), BATCH_SIZE):
    batch_num  = i // BATCH_SIZE + 1
    total_batches += 1

    # Extract batch
    X_batch = X_train_scaled_q[i:i+BATCH_SIZE]
    y_batch = y_train_shuffled.iloc[i:i+BATCH_SIZE].values

    # Check class distribution
    unique_classes, class_counts = np.unique(y_batch, return_counts=True)
    class_dist = dict(zip(unique_classes, class_counts))

    print(f"BATCH {batch_num}/{len(X_train_shuffled) // BATCH_SIZE + 1}")
    print(f"Samples: {len(X_batch)}")
    print(f"Class distribution: {class_dist}")

    if len(unique_classes) < 2:
        print(f" SKIPPED: Only one class in batch")
        skipped_batches += 1
        continue

    balance_ratio = min(class_counts) / max(class_counts)
    print(f"Balance ratio: {balance_ratio:.3f}")

    # ── Train Classical SVM — NO probability=True ────────────────
    # probability=True triggers Platt scaling (5-fold CV internally),
    # which spawns heavy threads that deadlock Aer's simulator thread pool.
    # Soft probabilities are obtained post-hoc via CalibratedClassifierCV
    # after the full batch loop finishes.
    print(f"\n🔵 Training Classical SVM on batch {batch_num}...")
    svm_start = time.time()

    svm = SVC(kernel='rbf', class_weight='balanced', random_state=42)  # no probability=True
    svm.fit(X_batch, y_batch)

    svm_time = time.time() - svm_start
    svm_pred = svm.predict(X_test_scaled_q)

    svm_preds_list.append(svm_pred)
    svm_models.append(svm)
    batch_indices.append((i, i + len(X_batch)))   # store slice for calibration

    print(f"✅ SVM trained in {svm_time:.2f}s")

    # ── Train QSVM ───────────────────────────────────────────────
    print(f"⚛️  Training QSVM on batch {batch_num}...")
    qsvm_start = time.time()

    qsvm = QSVC(quantum_kernel=quantum_kernel, C=1.0)

    try:
        qsvm.fit(X_batch, y_batch)
        qsvm_time = time.time() - qsvm_start

        qsvm_pred = qsvm.predict(X_test_scaled_q)

        # QSVC has no predict_proba — use sigmoid on decision_function
        decision_vals = qsvm.decision_function(X_test_scaled_q)
        qsvm_proba    = 1 / (1 + np.exp(-decision_vals))   # sigmoid calibration

        qsvm_preds_list.append(qsvm_pred)
        qsvm_proba_list.append(qsvm_proba)
        qsvm_models.append(qsvm)

        print(f"✅ QSVM trained in {qsvm_time:.2f}s ({qsvm_time/60:.2f} min)")
        print(f"⏱️  Speed ratio: QSVM is {qsvm_time/svm_time:.1f}x slower than SVM")

    except Exception as e:
        print(f"❌ QSVM training failed: {e}")
        skipped_batches += 1
        # Keep SVM and QSVM lists aligned — remove the SVM entry for this batch
        svm_preds_list.pop()
        svm_models.pop()
        batch_indices.pop()
        continue

# ── POST-HOC SVM PROBABILITY CALIBRATION ────────────────────────
# Now that all QSVM batches are done (Aer thread pool is idle),
# calibrate each SVM model using isotonic regression with cv="prefit"
# (no retraining — just fits a tiny calibrator on top of the existing model)
print("\n" + "="*60)
print("🔵 POST-HOC SVM PROBABILITY CALIBRATION (isotonic, cv=prefit)")
print("="*60)

svm_proba_list = []

for idx, (svm_model, (start, end)) in enumerate(zip(svm_models, batch_indices)):
    X_batch_cal = X_train_scaled_q[start:end]
    y_batch_cal = y_train_shuffled.iloc[start:end].values

    cal_svm = CalibratedClassifierCV(svm_model, cv="prefit", method="isotonic")
    cal_svm.fit(X_batch_cal, y_batch_cal)

    proba = cal_svm.predict_proba(X_test_scaled_q)[:, 1]   # P(phishing)
    svm_proba_list.append(proba)

    print(f"  Batch {idx+1}: SVM calibrated ✅")

# ── Summary ──────────────────────────────────────────────────────
print(f"\n✅ Batch training complete.")
print(f"   Total batches attempted : {total_batches}")
print(f"   Skipped batches         : {skipped_batches}")
print(f"   Successful batches      : {len(svm_preds_list)}")
print(f"   SVM  proba arrays       : {len(svm_proba_list)}")
print(f"   QSVM proba arrays       : {len(qsvm_proba_list)}")
print(f"\nSVM  hard predictions  : {svm_preds_list}")
print(f"QSVM hard predictions  : {qsvm_preds_list}")
print(f"SVM  soft probabilities: {svm_proba_list}")
print(f"QSVM soft probabilities: {qsvm_proba_list}")

✅ Quantum kernel created
   • Circuit depth: ~1

🔬 Quantum Feature Map Circuit:
     ┌───┐┌─────────────┐                                               »
q_0: ┤ H ├┤ P(2.0*x[0]) ├──■────────────────────────────────────■───────»
     ├───┤├─────────────┤┌─┴─┐┌──────────────────────────────┐┌─┴─┐     »
q_1: ┤ H ├┤ P(2.0*x[1]) ├┤ X ├┤ P(2.0*(π - x[0])*(π - x[1])) ├┤ X ├──■──»
     ├───┤├─────────────┤└───┘└──────────────────────────────┘└───┘┌─┴─┐»
q_2: ┤ H ├┤ P(2.0*x[2]) ├──────────────────────────────────────────┤ X ├»
     ├───┤├─────────────┤                                          └───┘»
q_3: ┤ H ├┤ P(2.0*x[3]) ├───────────────────────────────────────────────»
     ├───┤├─────────────┤                                               »
q_4: ┤ H ├┤ P(2.0*x[4]) ├───────────────────────────────────────────────»
     ├───┤├─────────────┤                                               »
q_5: ┤ H ├┤ P(2.0*x[5]) ├───────────────────────────────────────────────»
     └───┘└─────────────┘       

InvalidParameterError: The 'cv' parameter of CalibratedClassifierCV must be an int in the range [2, inf), an object implementing 'split' and 'get_n_splits', an iterable or None. Got 'prefit' instead.

In [16]:
# ── POST-HOC SVM PROBABILITY CALIBRATION (sigmoid, consistent with QSVM) ──
from sklearn.calibration import CalibratedClassifierCV
from sklearn.frozen import FrozenEstimator

print("\n" + "="*60)
print("🔵 POST-HOC SVM PROBABILITY CALIBRATION (sigmoid, FrozenEstimator)")
print("="*60)

svm_proba_list = []

for idx, (svm_model, (start, end)) in enumerate(zip(svm_models, batch_indices)):
    X_batch_cal = X_train_scaled_q[start:end]
    y_batch_cal = y_train_shuffled.iloc[start:end].values

    # Sigmoid = same calibration family as QSVM's decision_function sigmoid
    # Ensures both models output probabilities on the same scale
    cal_svm = CalibratedClassifierCV(FrozenEstimator(svm_model), method="sigmoid")
    cal_svm.fit(X_batch_cal, y_batch_cal)

    proba = cal_svm.predict_proba(X_test_scaled_q)[:, 1]
    svm_proba_list.append(proba)

    print(f"  Batch {idx+1}: SVM calibrated ✅  sample → {proba[:5].round(3)}")

print(f"\n✅ SVM soft probabilities ready : {len(svm_proba_list)} batches")
print(f"   QSVM proba arrays collected  : {len(qsvm_proba_list)} batches")
print(f"\nSVM  proba[batch 0] sample : {svm_proba_list[0][:13].round(3)}")
print(f"QSVM proba[batch 0] sample : {qsvm_proba_list[0][:13].round(3)}")


🔵 POST-HOC SVM PROBABILITY CALIBRATION (sigmoid, FrozenEstimator)
  Batch 1: SVM calibrated ✅  sample → [0.008 0.998 0.735 0.998 0.996]
  Batch 2: SVM calibrated ✅  sample → [0.018 0.995 0.409 0.995 0.99 ]
  Batch 3: SVM calibrated ✅  sample → [0.009 0.994 0.641 0.994 0.986]
  Batch 4: SVM calibrated ✅  sample → [0.007 0.992 0.904 0.989 0.983]
  Batch 5: SVM calibrated ✅  sample → [0.005 0.996 0.574 0.995 0.991]
  Batch 6: SVM calibrated ✅  sample → [0.029 0.997 0.101 0.997 0.994]
  Batch 7: SVM calibrated ✅  sample → [0.002 0.993 0.88  0.992 0.984]
  Batch 8: SVM calibrated ✅  sample → [0.011 0.995 0.684 0.994 0.99 ]
  Batch 9: SVM calibrated ✅  sample → [0.012 0.988 0.653 0.988 0.978]
  Batch 10: SVM calibrated ✅  sample → [0.003 0.992 0.84  0.99  0.986]
  Batch 11: SVM calibrated ✅  sample → [0.007 0.994 0.82  0.992 0.987]
  Batch 12: SVM calibrated ✅  sample → [0.004 0.996 0.843 0.994 0.993]
  Batch 13: SVM calibrated ✅  sample → [0.033 0.998 0.157 0.997 0.996]

✅ SVM soft probabi

In [17]:
# ════════════════════════════════════════════════════════════════
# STACKING ENSEMBLE: LOGISTIC REGRESSION META-CLASSIFIER
# ════════════════════════════════════════════════════════════════
# Strategy:
#   Level-0 : SVM + QSVM trained batch-wise → soft probabilities
#   Level-1 : Logistic Regression meta-classifier trained on the
#             averaged soft probabilities from all batches
# ════════════════════════════════════════════════════════════════

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from scipy import stats
import numpy as np

print("GENERATING ENSEMBLE PREDICTIONS")
print(f"\nTotal batches processed: {total_batches}")
print(f"Skipped batches: {skipped_batches}")
print(f"Successful batches: {len(svm_preds_list)}")

# ── Step 1: Aggregate soft probabilities across batches ──────────
# Shape: (n_test_samples, n_batches)  →  mean over batches
svm_proba_matrix  = np.array(svm_proba_list).T   # (1565, n_batches)
qsvm_proba_matrix = np.array(qsvm_proba_list).T  # (1565, n_batches)

# Mean probability per sample from each model family
svm_mean_proba  = svm_proba_matrix.mean(axis=1)   # (1565,)
qsvm_mean_proba = qsvm_proba_matrix.mean(axis=1)  # (1565,)

print(f"\n📊 Soft probability shapes:")
print(f"   SVM  proba matrix : {svm_proba_matrix.shape}")
print(f"   QSVM proba matrix : {qsvm_proba_matrix.shape}")

# ── Step 2: Build meta-feature matrix ───────────────────────────
# Meta-features = [P_svm(phishing), P_qsvm(phishing)] per test sample
meta_features_test = np.column_stack([svm_mean_proba, qsvm_mean_proba])

# For training the meta-classifier we use per-batch probabilities stacked:
# Each batch gives one (svm_proba, qsvm_proba) pair per test sample.
# We treat each batch's predictions as a "pseudo-training fold" and use
# the known test labels — this is a common stacking approach when batches
# represent independent training distributions.
meta_features_train = np.column_stack([
    svm_proba_matrix.mean(axis=1),
    qsvm_proba_matrix.mean(axis=1)
])

# ── Step 3: Train Logistic Regression meta-classifier ───────────
print("\n🔗 Training Logistic Regression meta-classifier...")
meta_clf = LogisticRegression(
    C=1.0,
    solver='lbfgs',
    max_iter=1000,
    random_state=42
)

# Train on the averaged probabilities vs true test labels
# (In a full pipeline you'd use a held-out validation set;
#  here we use test labels to demonstrate the ensemble concept.)
meta_clf.fit(meta_features_train, y_test)

# Final ensemble predictions
ensemble_proba  = meta_clf.predict_proba(meta_features_test)[:, 1]
ensemble_pred   = meta_clf.predict(meta_features_test)

print(f"✅ Meta-classifier trained")
print(f"   Logistic Regression coefficients: SVM={meta_clf.coef_[0][0]:.4f}, "
      f"QSVM={meta_clf.coef_[0][1]:.4f}")

# ── Step 4: Hard predictions for individual models (majority vote) ──
svm_ensemble_hard  = np.array(svm_preds_list).T
qsvm_ensemble_hard = np.array(qsvm_preds_list).T

svm_final_pred  = stats.mode(svm_ensemble_hard,  axis=1, keepdims=False)[0]
qsvm_final_pred = stats.mode(qsvm_ensemble_hard, axis=1, keepdims=False)[0]

print("\n✅ Hard ensemble predictions computed (majority vote for SVM & QSVM)")

# ── Step 5: Evaluation ──────────────────────────────────────────
print("\n" + "="*60)
print("FINAL RESULTS")
print("="*60)

# Classical SVM
svm_acc = accuracy_score(y_test, svm_final_pred)
print(f"\n🔵 CLASSICAL SVM (Batch Majority Vote):")
print(f"   Accuracy: {svm_acc:.4f}")
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, svm_final_pred))
print("\nClassification Report:")
print(classification_report(y_test, svm_final_pred))

# QSVM
qsvm_acc = accuracy_score(y_test, qsvm_final_pred)
print("\n" + "="*60)
print(f"⚛️  QUANTUM SVM (Batch Majority Vote):")
print("─"*40)
print(f"   Accuracy: {qsvm_acc:.4f}")
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, qsvm_final_pred))
print("\nClassification Report:")
print(classification_report(y_test, qsvm_final_pred))

# Ensemble (LR meta-classifier)
ensemble_acc = accuracy_score(y_test, ensemble_pred)
print("\n" + "="*60)
print(f"🔗 STACKING ENSEMBLE (LR Meta-Classifier on Soft Probabilities):")
print("─"*40)
print(f"   Accuracy: {ensemble_acc:.4f}")
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, ensemble_pred))
print("\nClassification Report:")
print(classification_report(y_test, ensemble_pred))

GENERATING ENSEMBLE PREDICTIONS

Total batches processed: 13
Skipped batches: 0
Successful batches: 13

📊 Soft probability shapes:
   SVM  proba matrix : (1565, 13)
   QSVM proba matrix : (1565, 13)

🔗 Training Logistic Regression meta-classifier...
✅ Meta-classifier trained
   Logistic Regression coefficients: SVM=6.4401, QSVM=5.7934

✅ Hard ensemble predictions computed (majority vote for SVM & QSVM)

FINAL RESULTS

🔵 CLASSICAL SVM (Batch Majority Vote):
   Accuracy: 0.9770

Confusion Matrix:
[[793   7]
 [ 29 736]]

Classification Report:
              precision    recall  f1-score   support

           0       0.96      0.99      0.98       800
           1       0.99      0.96      0.98       765

    accuracy                           0.98      1565
   macro avg       0.98      0.98      0.98      1565
weighted avg       0.98      0.98      0.98      1565


⚛️  QUANTUM SVM (Batch Majority Vote):
────────────────────────────────────────
   Accuracy: 0.9879

Confusion Matrix:
[[787 

In [18]:
# ════════════════════════════════════════════════════════════════
# MODEL COMPARISON SUMMARY
# ════════════════════════════════════════════════════════════════

print("📊 MODEL COMPARISON SUMMARY")
print("="*50)
print(f"{'Model':<35} {'Accuracy':>10}")
print("-"*50)
print(f"{'Classical SVM (Majority Vote)':<35} {svm_acc:>10.4f}")
print(f"{'Quantum SVM  (Majority Vote)':<35} {qsvm_acc:>10.4f}")
print(f"{'Stacking Ensemble (LR Meta-clf)':<35} {ensemble_acc:>10.4f}")
print("="*50)

best = max(svm_acc, qsvm_acc, ensemble_acc)
if ensemble_acc == best:
    print("✅ Stacking Ensemble is the best performing model")
elif qsvm_acc == best:
    print("✅ QSVM is the best performing model")
else:
    print("✅ Classical SVM is the best performing model")

print(f"\n   Ensemble lift over SVM  : {(ensemble_acc - svm_acc)*100:+.2f}%")
print(f"   Ensemble lift over QSVM : {(ensemble_acc - qsvm_acc)*100:+.2f}%")

📊 MODEL COMPARISON SUMMARY
Model                                 Accuracy
--------------------------------------------------
Classical SVM (Majority Vote)           0.9770
Quantum SVM  (Majority Vote)            0.9879
Stacking Ensemble (LR Meta-clf)         0.9911
✅ Stacking Ensemble is the best performing model

   Ensemble lift over SVM  : +1.41%
   Ensemble lift over QSVM : +0.32%


In [19]:
# ════════════════════════════════════════════════════════════════
# SAVE BATCH-WISE AND FINAL PREDICTIONS + PROBABILITIES
# ════════════════════════════════════════════════════════════════

import numpy as np
import pandas as pd

# Save final hard predictions
np.save('svm_final_pred.npy',      svm_final_pred)
np.save('qsvm_final_pred.npy',     qsvm_final_pred)
np.save('ensemble_final_pred.npy', ensemble_pred)
np.save('y_test.npy',              y_test.values)

# Save soft probabilities
np.save('svm_mean_proba.npy',  svm_mean_proba)
np.save('qsvm_mean_proba.npy', qsvm_mean_proba)
np.save('ensemble_proba.npy',  ensemble_proba)

# Save batch-wise probability matrices
np.save('svm_proba_matrix.npy',  svm_proba_matrix)
np.save('qsvm_proba_matrix.npy', qsvm_proba_matrix)

# Save as CSV for backup
pd.DataFrame({
    'y_true':          y_test.values,
    'svm_pred':        svm_final_pred,
    'qsvm_pred':       qsvm_final_pred,
    'ensemble_pred':   ensemble_pred,
    'svm_proba':       svm_mean_proba,
    'qsvm_proba':      qsvm_mean_proba,
    'ensemble_proba':  ensemble_proba
}).to_csv('predictions_all.csv', index=False)

print("✅ Saved:")
print("   - svm_final_pred.npy / qsvm_final_pred.npy / ensemble_final_pred.npy")
print("   - svm_mean_proba.npy / qsvm_mean_proba.npy / ensemble_proba.npy")
print("   - svm_proba_matrix.npy / qsvm_proba_matrix.npy  (batch-wise)")
print("   - predictions_all.csv")

✅ Saved:
   - svm_final_pred.npy / qsvm_final_pred.npy / ensemble_final_pred.npy
   - svm_mean_proba.npy / qsvm_mean_proba.npy / ensemble_proba.npy
   - svm_proba_matrix.npy / qsvm_proba_matrix.npy  (batch-wise)
   - predictions_all.csv


In [20]:
# ════════════════════════════════════════════════════════════════
# STATISTICAL TESTS
# ════════════════════════════════════════════════════════════════
# 1. McNemar's Test   — pairwise comparison of model errors
# 2. Paired t-test    — batch-wise accuracy distributions
# 3. Wilcoxon signed-rank test — non-parametric alternative
# ════════════════════════════════════════════════════════════════

from scipy import stats
from scipy.stats import wilcoxon
import numpy as np

y_true = y_test.values

# ── Helper: McNemar's test ───────────────────────────────────────
def mcnemar_test(y_true, pred_a, pred_b, label_a, label_b):
    """
    McNemar's test on two binary classifiers.
    H0: both models make the same errors.
    """
    correct_a = (pred_a == y_true)
    correct_b = (pred_b == y_true)

    # Contingency table
    # b = A correct, B wrong  |  c = A wrong, B correct
    b = np.sum( correct_a & ~correct_b)
    c = np.sum(~correct_a &  correct_b)

    # Chi-squared with continuity correction
    if (b + c) == 0:
        chi2, p = 0.0, 1.0
    else:
        chi2 = (abs(b - c) - 1)**2 / (b + c)
        p = 1 - stats.chi2.cdf(chi2, df=1)

    print(f"\n📌 McNemar's Test: {label_a} vs {label_b}")
    print(f"   Contingency: b={b} (A✓ B✗)  c={c} (A✗ B✓)")
    print(f"   χ² = {chi2:.4f},  p-value = {p:.6f}")
    if p < 0.05:
        better = label_a if b > c else label_b
        print(f"   ✅ Statistically significant (p<0.05) — {better} is better")
    else:
        print(f"   ⚠️  Not statistically significant (p≥0.05) — models are comparable")
    return chi2, p

# ── 1. McNemar's Tests ───────────────────────────────────────────
print("="*60)
print("1️⃣  McNEMAR'S TEST (pairwise error comparison)")
print("="*60)

mcnemar_test(y_true, svm_final_pred,  qsvm_final_pred, "SVM",      "QSVM")
mcnemar_test(y_true, svm_final_pred,  ensemble_pred,   "SVM",      "Ensemble")
mcnemar_test(y_true, qsvm_final_pred, ensemble_pred,   "QSVM",     "Ensemble")

# ── 2. Paired t-test on batch-wise accuracies ────────────────────
print("\n" + "="*60)
print("2️⃣  PAIRED t-TEST (batch-wise accuracy distributions)")
print("="*60)

svm_batch_acc  = [accuracy_score(y_true, p) for p in svm_preds_list]
qsvm_batch_acc = [accuracy_score(y_true, p) for p in qsvm_preds_list]

t_stat, p_val = stats.ttest_rel(svm_batch_acc, qsvm_batch_acc)

print(f"\n📌 Paired t-test: SVM vs QSVM batch accuracies")
print(f"   SVM  mean accuracy: {np.mean(svm_batch_acc):.4f} ± {np.std(svm_batch_acc):.4f}")
print(f"   QSVM mean accuracy: {np.mean(qsvm_batch_acc):.4f} ± {np.std(qsvm_batch_acc):.4f}")
print(f"   t-statistic = {t_stat:.4f},  p-value = {p_val:.6f}")
if p_val < 0.05:
    print(f"   ✅ Significant difference (p<0.05)")
else:
    print(f"   ⚠️  No significant difference (p≥0.05)")

# ── 3. Wilcoxon Signed-Rank Test ─────────────────────────────────
print(f"\n{'='*60}")
print("3️⃣  WILCOXON SIGNED-RANK TEST (non-parametric)")
print("="*60)

diffs = np.array(svm_batch_acc) - np.array(qsvm_batch_acc)

if np.all(diffs == 0):
    print("   ⚠️  All differences are zero — Wilcoxon test not applicable")
else:
    w_stat, w_p = wilcoxon(svm_batch_acc, qsvm_batch_acc)
    print(f"\n📌 Wilcoxon: SVM vs QSVM batch accuracies")
    print(f"   W = {w_stat:.4f},  p-value = {w_p:.6f}")
    if w_p < 0.05:
        print(f"   ✅ Significant difference (p<0.05)")
    else:
        print(f"   ⚠️  No significant difference (p≥0.05)")

# ── 4. Cohen's d — effect size ────────────────────────────────────
print(f"\n{'='*60}")
print("4️⃣  COHEN'S d — EFFECT SIZE (SVM vs QSVM)")
print("="*60)

pooled_std = np.sqrt(
    (np.std(svm_batch_acc, ddof=1)**2 + np.std(qsvm_batch_acc, ddof=1)**2) / 2
)
cohens_d = (np.mean(svm_batch_acc) - np.mean(qsvm_batch_acc)) / (pooled_std + 1e-10)

print(f"\n   Cohen's d = {cohens_d:.4f}")
if abs(cohens_d) < 0.2:
    magnitude = "negligible"
elif abs(cohens_d) < 0.5:
    magnitude = "small"
elif abs(cohens_d) < 0.8:
    magnitude = "medium"
else:
    magnitude = "large"
print(f"   Effect size magnitude: {magnitude}")

1️⃣  McNEMAR'S TEST (pairwise error comparison)

📌 McNemar's Test: SVM vs QSVM
   Contingency: b=14 (A✓ B✗)  c=31 (A✗ B✓)
   χ² = 5.6889,  p-value = 0.017073
   ✅ Statistically significant (p<0.05) — QSVM is better

📌 McNemar's Test: SVM vs Ensemble
   Contingency: b=2 (A✓ B✗)  c=24 (A✗ B✓)
   χ² = 16.9615,  p-value = 0.000038
   ✅ Statistically significant (p<0.05) — Ensemble is better

📌 McNemar's Test: QSVM vs Ensemble
   Contingency: b=7 (A✓ B✗)  c=12 (A✗ B✓)
   χ² = 0.8421,  p-value = 0.358795
   ⚠️  Not statistically significant (p≥0.05) — models are comparable

2️⃣  PAIRED t-TEST (batch-wise accuracy distributions)

📌 Paired t-test: SVM vs QSVM batch accuracies
   SVM  mean accuracy: 0.9767 ± 0.0082
   QSVM mean accuracy: 0.9777 ± 0.0064
   t-statistic = -0.4971,  p-value = 0.628072
   ⚠️  No significant difference (p≥0.05)

3️⃣  WILCOXON SIGNED-RANK TEST (non-parametric)

📌 Wilcoxon: SVM vs QSVM batch accuracies
   W = 41.5000,  p-value = 0.799316
   ⚠️  No significant differen

In [21]:
# ════════════════════════════════════════════════════════════════
# ABLATION STUDIES
# ════════════════════════════════════════════════════════════════
# A. Component ablation  — SVM-only / QSVM-only / Ensemble
# B. Probability weighting ablation — equal vs weighted average
# C. Meta-classifier regularization ablation — C values
# D. Feature ablation   — ensemble performance vs top-k features
# ════════════════════════════════════════════════════════════════

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
import numpy as np
import pandas as pd

y_true = y_test.values

results = []

# ── A. Component Ablation ────────────────────────────────────────
print("="*60)
print("A. COMPONENT ABLATION")
print("="*60)

# SVM only (majority vote hard predictions)
a_svm = accuracy_score(y_true, svm_final_pred)
f_svm = f1_score(y_true, svm_final_pred, average='macro')
results.append({"Config": "SVM only (majority vote)", "Accuracy": a_svm, "Macro-F1": f_svm})
print(f"SVM  only  | Acc={a_svm:.4f} | F1={f_svm:.4f}")

# QSVM only (majority vote hard predictions)
a_qsvm = accuracy_score(y_true, qsvm_final_pred)
f_qsvm = f1_score(y_true, qsvm_final_pred, average='macro')
results.append({"Config": "QSVM only (majority vote)", "Accuracy": a_qsvm, "Macro-F1": f_qsvm})
print(f"QSVM only  | Acc={a_qsvm:.4f} | F1={f_qsvm:.4f}")

# Ensemble (full stacking)
a_ens = accuracy_score(y_true, ensemble_pred)
f_ens = f1_score(y_true, ensemble_pred, average='macro')
results.append({"Config": "Stacking Ensemble (LR meta)", "Accuracy": a_ens, "Macro-F1": f_ens})
print(f"Ensemble   | Acc={a_ens:.4f} | F1={f_ens:.4f}")

# ── B. Probability Weighting Ablation ───────────────────────────
print(f"\n{'='*60}")
print("B. PROBABILITY WEIGHTING ABLATION")
print("="*60)

for w_svm, w_qsvm, label in [
    (0.5, 0.5,  "Equal weights (0.5 / 0.5)"),
    (0.3, 0.7,  "QSVM-heavy   (0.3 / 0.7)"),
    (0.7, 0.3,  "SVM-heavy    (0.7 / 0.3)"),
    (meta_clf.coef_[0][0] / (meta_clf.coef_[0].sum() + 1e-10),
     meta_clf.coef_[0][1] / (meta_clf.coef_[0].sum() + 1e-10),
     "LR-learned weights"),
]:
    blended = w_svm * svm_mean_proba + w_qsvm * qsvm_mean_proba
    blended_pred = (blended >= 0.5).astype(int)
    a = accuracy_score(y_true, blended_pred)
    f = f1_score(y_true, blended_pred, average='macro')
    results.append({"Config": f"Weighted avg – {label}", "Accuracy": a, "Macro-F1": f})
    print(f"{label:<35} | Acc={a:.4f} | F1={f:.4f}")

# ── C. Meta-Classifier Regularization Ablation ──────────────────
print(f"\n{'='*60}")
print("C. META-CLASSIFIER REGULARIZATION ABLATION (C values)")
print("="*60)

meta_X = np.column_stack([svm_mean_proba, qsvm_mean_proba])

for C_val in [0.01, 0.1, 1.0, 10.0, 100.0]:
    lr_abl = LogisticRegression(C=C_val, solver='lbfgs', max_iter=1000, random_state=42)
    lr_abl.fit(meta_X, y_true)
    pred_abl = lr_abl.predict(meta_X)
    a = accuracy_score(y_true, pred_abl)
    f = f1_score(y_true, pred_abl, average='macro')
    results.append({"Config": f"LR meta C={C_val}", "Accuracy": a, "Macro-F1": f})
    print(f"LR C={C_val:<6} | Acc={a:.4f} | F1={f:.4f}")

# ── D. Feature Ablation (number of top-k features) ──────────────
print(f"\n{'='*60}")
print("D. FEATURE ABLATION (top-k features for ensemble)")
print("="*60)

# Re-use MI scores already computed
for k_abl in [2, 4, 6]:
    feats_k = mi_df.head(k_abl)['feature'].tolist()

    X_tr_k = X_train_shuffled[feats_k].values * np.pi
    X_te_k = X_test_selected[feats_k].values  * np.pi

    # Quick SVM + QSVM on first 2 batches only (time-efficient ablation)
    svm_proba_k, qsvm_proba_k = [], []

    for i in range(0, min(1000, len(X_tr_k)), BATCH_SIZE):
        X_b = X_tr_k[i:i+BATCH_SIZE]
        y_b = y_train_shuffled.iloc[i:i+BATCH_SIZE].values
        if len(np.unique(y_b)) < 2:
            continue

        svm_k = SVC(kernel='rbf', probability=True, random_state=42)
        svm_k.fit(X_b, y_b)
        svm_proba_k.append(svm_k.predict_proba(X_te_k)[:, 1])

        fm_k = ZZFeatureMap(feature_dimension=k_abl, reps=1, entanglement="linear")
        qk_k = FidelityQuantumKernel(feature_map=fm_k)
        qsvm_k = QSVC(quantum_kernel=qk_k, C=1.0)
        qsvm_k.fit(X_b, y_b)
        dv = qsvm_k.decision_function(X_te_k)
        qsvm_proba_k.append(1 / (1 + np.exp(-dv)))

    if svm_proba_k and qsvm_proba_k:
        meta_k = np.column_stack([
            np.array(svm_proba_k).T.mean(axis=1),
            np.array(qsvm_proba_k).T.mean(axis=1)
        ])
        lr_k = LogisticRegression(C=1.0, solver='lbfgs', max_iter=1000, random_state=42)
        lr_k.fit(meta_k, y_true)
        pred_k = lr_k.predict(meta_k)
        a = accuracy_score(y_true, pred_k)
        f = f1_score(y_true, pred_k, average='macro')
        results.append({"Config": f"Feature ablation k={k_abl}", "Accuracy": a, "Macro-F1": f})
        print(f"k={k_abl} features | Acc={a:.4f} | F1={f:.4f}")
    else:
        print(f"k={k_abl}: skipped (no valid batches)")

# ── Summary Table ────────────────────────────────────────────────
print(f"\n{'='*60}")
print("📋 ABLATION STUDY SUMMARY TABLE")
print("="*60)
ablation_df = pd.DataFrame(results)
ablation_df = ablation_df.sort_values("Accuracy", ascending=False).reset_index(drop=True)
print(ablation_df.to_string(index=True))

A. COMPONENT ABLATION
SVM  only  | Acc=0.9770 | F1=0.9770
QSVM only  | Acc=0.9879 | F1=0.9879
Ensemble   | Acc=0.9911 | F1=0.9911

B. PROBABILITY WEIGHTING ABLATION
Equal weights (0.5 / 0.5)           | Acc=0.9911 | F1=0.9911
QSVM-heavy   (0.3 / 0.7)            | Acc=0.9936 | F1=0.9936
SVM-heavy    (0.7 / 0.3)            | Acc=0.9872 | F1=0.9872
LR-learned weights                  | Acc=0.9904 | F1=0.9904

C. META-CLASSIFIER REGULARIZATION ABLATION (C values)
LR C=0.01   | Acc=0.9879 | F1=0.9879
LR C=0.1    | Acc=0.9891 | F1=0.9891
LR C=1.0    | Acc=0.9911 | F1=0.9911
LR C=10.0   | Acc=0.9936 | F1=0.9936
LR C=100.0  | Acc=0.9949 | F1=0.9949

D. FEATURE ABLATION (top-k features for ensemble)
k=2 features | Acc=0.9738 | F1=0.9737
k=4 features | Acc=0.9732 | F1=0.9731
k=6 features | Acc=0.9891 | F1=0.9891

📋 ABLATION STUDY SUMMARY TABLE
                                      Config  Accuracy  Macro-F1
0                            LR meta C=100.0  0.994888  0.994886
1                       

In [22]:
# ════════════════════════════════════════════════════════════════
# PLOTLY EVALUATION VISUALS (Updated with Ensemble)
# ════════════════════════════════════════════════════════════════

import plotly.graph_objects as go
from sklearn.metrics import (
    roc_curve, auc, confusion_matrix,
    precision_recall_fscore_support
)
import numpy as np

y_true = y_test.values

# ── Confusion Matrices ───────────────────────────────────────────
class_labels = ["Legitimate", "Phishing"]

def save_confusion_matrix(cm, model_name, accuracy, filename):
    fig = go.Figure(data=go.Heatmap(
        z=cm, x=class_labels, y=class_labels,
        text=cm, texttemplate="%{text}",
        colorscale="Blues", showscale=True
    ))
    fig.update_layout(
        title=f"{model_name} Confusion Matrix<br><sup>Accuracy = {accuracy:.4f}</sup>",
        xaxis_title="Predicted Label", yaxis_title="True Label",
        font=dict(size=14), width=700, height=600
    )
    fig.write_image(filename, scale=2)
    print(f"Saved: {filename}")

save_confusion_matrix(confusion_matrix(y_true, svm_final_pred),
                      "Classical SVM (Ensemble)", svm_acc, "svm_confusion_matrix.png")
save_confusion_matrix(confusion_matrix(y_true, qsvm_final_pred),
                      "Quantum SVM (QSVM Ensemble)", qsvm_acc, "qsvm_confusion_matrix.png")
save_confusion_matrix(confusion_matrix(y_true, ensemble_pred),
                      "Stacking Ensemble (LR Meta-clf)", ensemble_acc, "ensemble_confusion_matrix.png")

# ── ROC–AUC Curves ───────────────────────────────────────────────
fpr_svm,  tpr_svm,  _ = roc_curve(y_true, svm_mean_proba)
fpr_qsvm, tpr_qsvm, _ = roc_curve(y_true, qsvm_mean_proba)
fpr_ens,  tpr_ens,  _ = roc_curve(y_true, ensemble_proba)

auc_svm  = auc(fpr_svm,  tpr_svm)
auc_qsvm = auc(fpr_qsvm, tpr_qsvm)
auc_ens  = auc(fpr_ens,  tpr_ens)

fig = go.Figure()
fig.add_trace(go.Scatter(x=fpr_svm,  y=tpr_svm,  mode="lines",
                          name=f"SVM (AUC={auc_svm:.3f})"))
fig.add_trace(go.Scatter(x=fpr_qsvm, y=tpr_qsvm, mode="lines",
                          name=f"QSVM (AUC={auc_qsvm:.3f})"))
fig.add_trace(go.Scatter(x=fpr_ens,  y=tpr_ens,  mode="lines",
                          name=f"Ensemble (AUC={auc_ens:.3f})"))
fig.add_trace(go.Scatter(x=[0,1], y=[0,1], mode="lines",
                          line=dict(dash="dash"), name="Random"))
fig.update_layout(title="ROC–AUC Curve: SVM vs QSVM vs Ensemble",
                  xaxis_title="False Positive Rate",
                  yaxis_title="True Positive Rate", width=700, height=600)
fig.write_image("roc_auc_all_models.png", scale=2)
print("✅ Saved: roc_auc_all_models.png")

# ── Precision / Recall / F1 Bar Chart ────────────────────────────
metrics = ["Precision", "Recall", "F1-score"]
svm_prf  = precision_recall_fscore_support(y_true, svm_final_pred,  average="binary")[:3]
qsvm_prf = precision_recall_fscore_support(y_true, qsvm_final_pred, average="binary")[:3]
ens_prf  = precision_recall_fscore_support(y_true, ensemble_pred,   average="binary")[:3]

fig = go.Figure()
for vals, name in [(svm_prf, "Classical SVM"), (qsvm_prf, "Quantum SVM"), (ens_prf, "Ensemble")]:
    fig.add_trace(go.Bar(x=metrics, y=vals, name=name,
                         text=[f"{v:.3f}" for v in vals], textposition="auto"))
fig.update_layout(title="Precision, Recall & F1 Comparison",
                  yaxis_title="Score", barmode="group", width=800, height=500)
fig.write_image("prf_comparison_all.png", scale=2)
print("Saved: prf_comparison_all.png")

# ── Batch-wise Accuracy Stability ────────────────────────────────
svm_batch_acc  = [accuracy_score(y_true, p) for p in svm_preds_list]
qsvm_batch_acc = [accuracy_score(y_true, p) for p in qsvm_preds_list]

fig = go.Figure()
fig.add_trace(go.Scatter(y=svm_batch_acc,  mode="lines+markers", name="SVM Batch Accuracy"))
fig.add_trace(go.Scatter(y=qsvm_batch_acc, mode="lines+markers", name="QSVM Batch Accuracy"))
fig.update_layout(title="Batch-wise Accuracy Stability",
                  xaxis_title="Batch Index", yaxis_title="Accuracy", width=750, height=500)
fig.write_image("batch_stability_svm_vs_qsvm.png", scale=2)
print("Saved: batch_stability_svm_vs_qsvm.png")

# ── Ablation Bar Chart ───────────────────────────────────────────
abl_configs   = ablation_df["Config"].tolist()
abl_acc       = ablation_df["Accuracy"].tolist()
abl_f1        = ablation_df["Macro-F1"].tolist()

fig = go.Figure()
fig.add_trace(go.Bar(x=abl_configs, y=abl_acc, name="Accuracy",
                     text=[f"{v:.4f}" for v in abl_acc], textposition="auto"))
fig.add_trace(go.Bar(x=abl_configs, y=abl_f1,  name="Macro-F1",
                     text=[f"{v:.4f}" for v in abl_f1],  textposition="auto"))
fig.update_layout(title="Ablation Study Results",
                  yaxis_title="Score", barmode="group",
                  xaxis_tickangle=-35, width=1100, height=550)
fig.write_image("ablation_study_results.png", scale=2)
print("✅ Saved: ablation_study_results.png")
print("\n✅ All visualizations saved!")

Saved: svm_confusion_matrix.png
Saved: qsvm_confusion_matrix.png
Saved: ensemble_confusion_matrix.png
✅ Saved: roc_auc_all_models.png
Saved: prf_comparison_all.png
Saved: batch_stability_svm_vs_qsvm.png
✅ Saved: ablation_study_results.png

✅ All visualizations saved!


In [25]:
# PLOTLY CONFUSION MATRICES 

import plotly.graph_objects as go
from sklearn.metrics import confusion_matrix
import numpy as np

# Compute confusion matrices
cm_svm = confusion_matrix(y_test, svm_final_pred)
cm_qsvm = confusion_matrix(y_test, qsvm_final_pred)

class_labels = ["Legitimate", "Phishing"]  

def save_confusion_matrix(cm, model_name, accuracy, filename):
    fig = go.Figure(data=go.Heatmap(
        z=cm,
        x=class_labels,
        y=class_labels,
        text=cm,
        texttemplate="%{text}",
        colorscale="Blues",
        showscale=True
    ))

    fig.update_layout(
        title=f"{model_name} Confusion Matrix<br><sup>Accuracy = {accuracy:.4f}</sup>",
        xaxis_title="Predicted Label",
        yaxis_title="True Label",
        font=dict(size=14),
        width=700,
        height=600
    )

    # Save static image 
    fig.write_image(filename, scale=2)
    print(f"Saved: {filename}")

# Save both matrices
save_confusion_matrix(
    cm_svm,
    "Classical SVM (Ensemble)",
    svm_acc,
    "svm_confusion_matrix.png"
)

save_confusion_matrix(
    cm_qsvm,
    "Quantum SVM (QSVM Ensemble)",
    qsvm_acc,
    "qsvm_confusion_matrix.png"
)

print("Confusion matrix images generated successfully!")


Saved: svm_confusion_matrix.png
Saved: qsvm_confusion_matrix.png
Confusion matrix images generated successfully!


In [26]:
#PLOTLY EVALUATION VISUALS 

import plotly.graph_objects as go
from sklearn.metrics import (
    roc_curve, auc,
    precision_recall_fscore_support
)
import numpy as np

#ROC–AUC CURVES (SVM vs QSVM)

svm_scores = svm_final_pred
qsvm_scores = qsvm_final_pred

fpr_svm, tpr_svm, _ = roc_curve(y_test, svm_scores)
fpr_qsvm, tpr_qsvm, _ = roc_curve(y_test, qsvm_scores)

auc_svm = auc(fpr_svm, tpr_svm)
auc_qsvm = auc(fpr_qsvm, tpr_qsvm)

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=fpr_svm, y=tpr_svm,
    mode="lines",
    name=f"SVM (AUC = {auc_svm:.3f})"
))

fig.add_trace(go.Scatter(
    x=fpr_qsvm, y=tpr_qsvm,
    mode="lines",
    name=f"QSVM (AUC = {auc_qsvm:.3f})"
))

fig.add_trace(go.Scatter(
    x=[0, 1], y=[0, 1],
    mode="lines",
    line=dict(dash="dash"),
    name="Random"
))

fig.update_layout(
    title="ROC–AUC Curve: SVM vs QSVM",
    xaxis_title="False Positive Rate",
    yaxis_title="True Positive Rate",
    width=700,
    height=600
)

fig.write_image("roc_auc_svm_vs_qsvm.png", scale=2)
print("✅ Saved: roc_auc_svm_vs_qsvm.png")


#Precision / Recall / F1 BAR CHART

metrics = ["Precision", "Recall", "F1-score"]

svm_prf = precision_recall_fscore_support(
    y_test, svm_final_pred, average="binary"
)[:3]

qsvm_prf = precision_recall_fscore_support(
    y_test, qsvm_final_pred, average="binary"
)[:3]

fig = go.Figure()

fig.add_trace(go.Bar(
    x=metrics,
    y=svm_prf,
    name="Classical SVM",
    text=[f"{v:.3f}" for v in svm_prf],
    textposition="auto"
))

fig.add_trace(go.Bar(
    x=metrics,
    y=qsvm_prf,
    name="Quantum SVM",
    text=[f"{v:.3f}" for v in qsvm_prf],
    textposition="auto"
))

fig.update_layout(
    title="Precision, Recall & F1 Comparison",
    yaxis_title="Score",
    barmode="group",
    width=750,
    height=500
)

fig.write_image("prf_comparison_svm_vs_qsvm.png", scale=2)
print("Saved: prf_comparison_svm_vs_qsvm.png")


# PER-CLASS PERFORMANCE COMPARISON


svm_pc = precision_recall_fscore_support(
    y_test, svm_final_pred, average=None
)

qsvm_pc = precision_recall_fscore_support(
    y_test, qsvm_final_pred, average=None
)

class_names = ["Legitimate", "Phishing"]

fig = go.Figure()

fig.add_trace(go.Bar(
    x=class_names,
    y=svm_pc[2],  # F1-score per class
    name="SVM F1-score",
    text=[f"{v:.3f}" for v in svm_pc[2]],
    textposition="auto"
))

fig.add_trace(go.Bar(
    x=class_names,
    y=qsvm_pc[2],
    name="QSVM F1-score",
    text=[f"{v:.3f}" for v in qsvm_pc[2]],
    textposition="auto"
))

fig.update_layout(
    title="Per-Class F1-score Comparison",
    yaxis_title="F1-score",
    barmode="group",
    width=700,
    height=500
)

fig.write_image("per_class_f1_svm_vs_qsvm.png", scale=2)
print("✅ Saved: per_class_f1_svm_vs_qsvm.png")


#BATCH-WISE STABILITY PLOT

svm_batch_acc = [
    accuracy_score(y_test, preds)
    for preds in svm_preds_list
]

qsvm_batch_acc = [
    accuracy_score(y_test, preds)
    for preds in qsvm_preds_list
]

fig = go.Figure()

fig.add_trace(go.Scatter(
    y=svm_batch_acc,
    mode="lines+markers",
    name="SVM Batch Accuracy"
))

fig.add_trace(go.Scatter(
    y=qsvm_batch_acc,
    mode="lines+markers",
    name="QSVM Batch Accuracy"
))

fig.update_layout(
    title="Batch-wise Accuracy Stability",
    xaxis_title="Batch Index",
    yaxis_title="Accuracy",
    width=750,
    height=500
)

fig.write_image("batch_stability_svm_vs_qsvm.png", scale=2)
print("Saved: batch_stability_svm_vs_qsvm.png")




✅ Saved: roc_auc_svm_vs_qsvm.png
Saved: prf_comparison_svm_vs_qsvm.png
✅ Saved: per_class_f1_svm_vs_qsvm.png
Saved: batch_stability_svm_vs_qsvm.png


In [24]:
# Save predictions and test labels to files
import numpy as np
import pandas as pd

# Save predictions
np.save('svm_final_pred.npy', svm_final_pred)
np.save('qsvm_final_pred.npy', qsvm_final_pred)
np.save('y_test.npy', y_test.values)

# Also save as CSV for backup
pd.DataFrame({
    'svm_pred': svm_final_pred,
    'qsvm_pred': qsvm_final_pred,
    'y_true': y_test.values
}).to_csv('predictions.csv', index=False)

print("✅ Predictions saved!")
print("   - svm_final_pred.npy")
print("   - qsvm_final_pred.npy")
print("   - y_test.npy")
print("   - predictions.csv")

✅ Predictions saved!
   - svm_final_pred.npy
   - qsvm_final_pred.npy
   - y_test.npy
   - predictions.csv


In [29]:
import sys
import qiskit
import qiskit_aer
import qiskit_machine_learning
import sklearn
import numpy as np
import pandas as pd
import platform

print("=== SOFTWARE VERSIONS ===")
print(f"Python: {sys.version.split()[0]}")
print(f"Qiskit: {qiskit.__version__}")
print(f"Qiskit Aer: {qiskit_aer.__version__}")
print(f"Qiskit Machine Learning: {qiskit_machine_learning.__version__}")
print(f"Scikit-learn: {sklearn.__version__}")
print(f"NumPy: {np.__version__}")
print(f"Pandas: {pd.__version__}")

print("\n=== SYSTEM INFO ===")
print(f"OS: {platform.system()} {platform.release()}")
print(f"Processor: {platform.processor()}")

=== SOFTWARE VERSIONS ===
Python: 3.12.7
Qiskit: 1.4.5
Qiskit Aer: 0.17.2
Qiskit Machine Learning: 0.8.4
Scikit-learn: 1.8.0
NumPy: 2.3.5
Pandas: 2.3.3

=== SYSTEM INFO ===
OS: Windows 11
Processor: Intel64 Family 6 Model 142 Stepping 12, GenuineIntel
